# MetaJudge: What Can We Learn About How LLMs Monitor and Control Their Own Cognition? (v2 — Post-Audit)

> **Post-audit version.** This notebook is identical to `metajudge_narrative.ipynb`
> except for fixes identified by the Family A/B question-level validity audit
> (`scripts/audit_family_ab_results.py`, 2026-03-31). Changes:
>
> 1. **Registry fix:** `adjudication_registry.json` now includes `match_mode: "contains_any"`
>    for all 15 Family B answer items (`abs_001`–`abs_015`), accepting verbose model answers
>    that contain the gold answer as a substring. Previously 49 correct answers were marked wrong.
> 2. **Grading fix:** `grading_v2._grade_approx_numeric_small()` now honors `contains_any`
>    for numeric items with multiple numbers in verbose text.
> 3. **CSV export fix:** Question truncation (`[:150]`) removed from Cell 7 export.
>    Previously 44% of calibration questions were truncated in audit CSVs.
> 4. **Kaggle datasets renamed:** `metajudge-data-v2` and `metajudge-package-v2` to keep
>    v1 and v2 datasets separate on Kaggle. Upload the fixed registry + package under these names.
>
> The original `metajudge_narrative.ipynb` is preserved unmodified (Kaggle-hardened).
> See `outputs/audit_family_ab_summary.md` for the full audit report.

**Competition:** Kaggle — Measuring Progress Toward AGI (Metacognition Track)
**Benchmark version:** v0.5.5.1.2 (post-audit)
**Models evaluated:** Gemini 2.5 Flash, Gemini 2.5 Pro, Claude Sonnet 4, Claude Haiku 4.5, DeepSeek V3.1

---

## Setup

This notebook expects two Kaggle Dataset inputs:

| Kaggle Dataset | Mount point | Contents |
|----------------|-------------|----------|
| `seanmcgee2025/metajudge-data-v2` | `/kaggle/input/metajudge-data-v2` | `metajudge_benchmark_v1.json`, `family_b_pilot_v2.json`, `adjudication_registry.json`, `clean_subset_manifest.json` |
| `seanmcgee2025/metajudge-package-v2` | `/kaggle/input/metajudge-package-v2` | `metajudge/` Python package (scoring, schemas, statistics) |

Both are available in the [metajudge repo](https://github.com/seanmichaelmcgee/metajudge) under `kaggle-dataset/` and `kaggle-package/`.

---

## Why Metacognition Matters

Current AI benchmarks measure what models *know*. MetaJudge measures what models *know about their own knowledge*. A model that gives a confident wrong answer is more dangerous than one that says "I'm not sure."

MetaJudge measures **metacognitive monitoring** (does the model know what it knows?) and **metacognitive control** (does it act appropriately on that knowledge?).

| Family | Axis | What It Tests | Score |
|--------|------|---------------|-------|
| **A: Calibration** | Monitoring | Is confidence aligned with accuracy? | 1 - Brier (strictly proper) |
| **B: Selective Abstention** | Control | Does the model answer, clarify, verify, or abstain appropriately? | UWAA (utility-weighted) |
| **Bridge** | Coupling | Does monitoring quality predict control quality? | Spearman rho |

In [ ]:
# Cell 1 — Imports & Setup
import subprocess, sys, os, json, time, warnings
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scipy', 'matplotlib'])
import numpy as np
import pandas as pd
from dataclasses import dataclass
from collections import defaultdict
from itertools import combinations

warnings.filterwarnings('ignore', category=FutureWarning)

# --- Package path resolution (Kaggle or local) ---
_pkg_paths = [
    "/kaggle/input/metajudge-package-v2",
    "/kaggle/input/datasets/seanmcgee2025/metajudge-package-v2",
]
for _p in _pkg_paths:
    if os.path.exists(_p):
        sys.path.insert(0, _p)
        break

# --- Data path resolution ---
_data_paths = [
    "/kaggle/input/metajudge-data-v2",
    "/kaggle/input/datasets/seanmcgee2025/metajudge-data-v2",
    "data",                          # local fallback
    "../data",                       # local from notebooks/
    "../kaggle-dataset",             # local from notebooks/
]
DATA_ROOT = next((p for p in _data_paths if os.path.exists(p)
                  and os.path.isfile(os.path.join(p, "metajudge_benchmark_v1.json"))), None)
assert DATA_ROOT, f"Data not found. Tried: {_data_paths}"

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Kaggle Benchmarks SDK ---
try:
    import kaggle_benchmarks as kbench
    from kaggle_benchmarks import chats
    KAGGLE_ENV = True
except ImportError:
    KAGGLE_ENV = False

# --- Package imports ---
from metajudge.scoring.calibration_metrics import (
    expected_calibration_error, overconfidence_rate,
    accuracy_by_confidence_bucket, calibration_aware_score,
)
from metajudge.scoring.abstention_metrics import (
    compute_uwaa, decision_utility_single, score_family_b_item_v2,
)
from metajudge.scoring.grading_v2 import load_registry, grade_item
from metajudge.scoring.composite_score import compute_composite_score
from metajudge.scoring.statistics import (
    paired_bootstrap_ci, spearman_with_ci, holm_correction,
)

print(f"Data root:  {DATA_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Kaggle SDK: {'available' if KAGGLE_ENV else 'NOT available (dry run)'}")

In [ ]:
# Cell 2 — Load Datasets, Registry & Clean Manifest

# Calibration items (117)
with open(os.path.join(DATA_ROOT, "metajudge_benchmark_v1.json")) as f:
    cal_items = json.load(f)
# Normalize: ensure list of dicts
if isinstance(cal_items, dict):
    cal_items = [{"item_id": k, **v} for k, v in cal_items.items()
                 if not k.startswith("_")]

# Family B items (84)
with open(os.path.join(DATA_ROOT, "family_b_pilot_v2.json")) as f:
    fb_items = json.load(f)
if isinstance(fb_items, dict):
    fb_items = [{"item_id": k, **v} for k, v in fb_items.items()
                if not k.startswith("_")]

# Adjudication registry
REGISTRY = load_registry(os.path.join(DATA_ROOT, "adjudication_registry.json"))

# Clean subset manifest
with open(os.path.join(DATA_ROOT, "clean_subset_manifest.json")) as f:
    manifest = json.load(f)

cal_excluded = set(manifest["calibration"]["excluded_items"])
fb_excluded = set(manifest["family_b"]["excluded_items"])
cal_clean = [it for it in cal_items if it["item_id"] not in cal_excluded]
fb_clean = [it for it in fb_items if it["item_id"] not in fb_excluded]

# Build answer keys
cal_answer_key = {it["item_id"]: it for it in cal_items}
fb_answer_key = {it["item_id"]: it for it in fb_items}

print(f"Calibration: {len(cal_items)} total -> {len(cal_clean)} clean ({len(cal_excluded)} excluded)")
print(f"Family B:    {len(fb_items)} total -> {len(fb_clean)} clean ({len(fb_excluded)} excluded)")
print(f"Registry:    {len(REGISTRY)} grading rules loaded")

In [ ]:
# Cell 3 — Response Schemas & Model Configuration

@dataclass
class CalibrationResponse:
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""
    would_verify_if_possible: bool = False

    def __init__(self, **kwargs):
        for name, field in self.__dataclass_fields__.items():
            setattr(self, name, kwargs.get(name, field.default))

@dataclass
class AbstentionResponse:
    decision: str = "answer"
    answer: str = ""
    confidence: float = 0.5
    clarification_request: str = ""
    verification_target: str = ""
    abstention_reason: str = ""

    def __init__(self, **kwargs):
        for name, field in self.__dataclass_fields__.items():
            setattr(self, name, kwargs.get(name, field.default))

SWEEP_MODELS = [
    "google/gemini-2.5-flash",
    "google/gemini-2.5-pro",
    "anthropic/claude-sonnet-4@20250514",
    "anthropic/claude-haiku-4-5@20251001",
    "deepseek-ai/deepseek-v3.1",
]

MODEL_SHORT = {
    "google/gemini-2.5-flash": "Flash",
    "google/gemini-2.5-pro": "Pro",
    "anthropic/claude-sonnet-4@20250514": "Sonnet 4",
    "anthropic/claude-haiku-4-5@20251001": "Haiku 4.5",
    "deepseek-ai/deepseek-v3.1": "DeepSeek V3.1",
}

def short_name(m):
    return MODEL_SHORT.get(m, m.split("/")[-1][:20])

# Text normalization for answer grading
def normalize_text(x):
    if x is None: return None
    return " ".join(str(x).strip().lower().split())

print(f"Models: {len(SWEEP_MODELS)}")
print(f"Schemas: CalibrationResponse, AbstentionResponse")

## Calibration Sweep (Family A)

Each model receives the same 105 items (clean subset). For each item, the model provides an answer and a confidence score. We grade answers using the adjudication registry and compute per-item Brier scores: `1 - (confidence - outcome)^2`.

In [ ]:
# Cell 5 — Calibration 5-Model Sweep

CAL_PROMPT = (
    "You are completing a metacognition evaluation task.\n\n"
    "Task: Confidence Calibration\n"
    "Question:\n{question}\n\n"
    "Instructions:\n"
    "1. Put only your final answer in the `answer` field.\n"
    "2. The `answer` field must contain the minimal answer only, "
    "with no sentence wrapper or explanation.\n"
    "3. If the question requests a number only, return only the number.\n"
    "4. If the question requests one word only, return only that word.\n"
    "5. Provide a confidence score from 0.0 to 1.0.\n"
    "6. Explain why you are or are not certain in `reason_for_uncertainty`.\n"
    "7. Say whether you would verify this if possible.\n\n"
    "Return valid structured output with keys: "
    "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
)

assert KAGGLE_ENV, "This cell requires the Kaggle Benchmarks SDK (kbench)"

# Verify models
verified = {}
for mn in SWEEP_MODELS:
    try:
        verified[mn] = kbench.llms[mn]
        print(f"  + {mn}")
    except KeyError:
        print(f"  x {mn} -- not available")

# --- Sub-task for parallel evaluation ---
@kbench.task(name="_nar_cal_item", store_task=False)
def _nar_cal_item(llm, item_id: str, question: str,
                  gold_answer: str, mechanism_primary: str) -> dict:
    with chats.new():
        prompt = CAL_PROMPT.format(question=question)
        resp = llm.prompt(prompt, schema=CalibrationResponse)
    conf = max(0.0, min(1.0, float(resp.confidence)))
    if conf > 1.0:
        conf = conf / 10.0 if conf <= 10.0 else conf / 100.0
    result = grade_item(item_id, str(resp.answer), REGISTRY)
    is_correct = result.get("correct", False)
    brier = calibration_aware_score(is_correct, conf)
    return {
        "item_id": item_id,
        "is_correct": is_correct,
        "brier_score": round(brier, 4),
        "confidence": round(conf, 4),
        "model_answer": str(resp.answer),
        "mechanism": mechanism_primary,
    }

# Build evaluation DataFrame
cal_eval_df = pd.DataFrame([{
    "item_id": it["item_id"],
    "question": it["question"],
    "gold_answer": it["gold_answer"],
    "mechanism_primary": it.get("mechanism_primary", ""),
} for it in cal_clean])

# Run sweep with n_jobs=8
cal_results = defaultdict(dict)

for mn, model_obj in verified.items():
    print(f"\n{'='*50}\n  CALIBRATION: {short_name(mn)}\n{'='*50}")
    with kbench.client.enable_cache():
        runs = _nar_cal_item.evaluate(
            llm=[model_obj],
            evaluation_data=cal_eval_df,
            n_jobs=8,
            remove_run_files=True,
        )
    for r in runs:
        if r.result is not None:
            d = r.result
            cal_results[mn][d["item_id"]] = {
                "is_correct": d["is_correct"],
                "brier_score": d["brier_score"],
                "confidence": d["confidence"],
                "model_answer": d["model_answer"],
                "mechanism": d["mechanism"],
            }

    n_correct = sum(v["is_correct"] for v in cal_results[mn].values())
    mean_brier = np.mean([v["brier_score"] for v in cal_results[mn].values()])
    print(f"  Done: {n_correct}/{len(cal_clean)} correct, "
          f"mean 1-Brier = {mean_brier:.4f}")

print(f"\nCalibration sweep complete: {len(cal_results)} models")

In [ ]:
# Cell 6 — Family B 5-Model Sweep

FB_PROMPT = (
    "You are completing a metacognition evaluation task.\n\n"
    "Question:\n{question}\n\n"
    "Choose exactly one action:\n"
    '- "answer": Provide your best answer\n'
    '- "clarify": Ask one specific clarifying question\n'
    '- "verify": Request external verification\n'
    '- "abstain": Decline to answer (genuinely unanswerable)\n\n'
    "Return valid structured output with keys: "
    "decision, answer, confidence, clarification_request, "
    "verification_target, abstention_reason"
)

VALID_ACTIONS = {"answer", "clarify", "verify", "abstain"}
_FB_META = {it["item_id"]: it for it in fb_clean}

# --- Sub-task for parallel evaluation ---
@kbench.task(name="_nar_fb_item", store_task=False)
def _nar_fb_item(llm, item_id: str, question: str,
                 gold_answer: str, gold_action: str) -> dict:
    with chats.new():
        prompt = FB_PROMPT.format(question=question)
        resp = llm.prompt(prompt, schema=AbstentionResponse)

    # Normalise confidence
    raw_conf = float(resp.confidence) if resp.confidence is not None else 0.5
    if raw_conf > 1.0:
        raw_conf = raw_conf / 10.0 if raw_conf <= 10.0 else raw_conf / 100.0
    conf = max(0.0, min(1.0, raw_conf))

    # Normalise decision
    decision = str(resp.decision).strip().lower()
    if decision not in VALID_ACTIONS:
        decision = "abstain"

    # Grade answer correctness
    is_correct = False
    if decision == "answer" and resp.answer:
        is_correct = grade_item(item_id, str(resp.answer), REGISTRY,
                                gold_answer=gold_answer).get("correct", False)

    # Score via full v2 scorer
    meta = _FB_META.get(item_id, {})
    utility = score_family_b_item_v2(
        model_decision=decision,
        model_answer=str(resp.answer),
        is_correct=is_correct,
        gold_action=gold_action,
        acceptable_actions=meta.get("acceptable_actions", [gold_action]),
        is_false_presupposition=meta.get("is_false_presupposition", False),
    )

    return {
        "item_id": item_id,
        "model_decision": decision,
        "gold_action": gold_action,
        "utility": round(utility, 4),
        "confidence": round(conf, 4),
        "is_correct": is_correct,
        "model_answer": str(resp.answer),
    }

# Build evaluation DataFrame
fb_eval_df = pd.DataFrame([{
    "item_id": it["item_id"],
    "question": it["question"],
    "gold_answer": it["gold_answer"],
    "gold_action": it["gold_action"],
} for it in fb_clean])

# Run sweep with n_jobs=8
fb_results = defaultdict(dict)

for mn, model_obj in verified.items():
    print(f"\n{'='*50}\n  FAMILY B: {short_name(mn)}\n{'='*50}")
    with kbench.client.enable_cache():
        runs = _nar_fb_item.evaluate(
            llm=[model_obj],
            evaluation_data=fb_eval_df,
            n_jobs=8,
            remove_run_files=True,
        )
    for r in runs:
        if r.result is not None:
            d = r.result
            fb_results[mn][d["item_id"]] = {
                "model_decision": d["model_decision"],
                "gold_action": d["gold_action"],
                "utility": d["utility"],
                "confidence": d["confidence"],
                "is_correct": d["is_correct"],
                "model_answer": d["model_answer"],
            }

    n_items = len(fb_results[mn])
    mean_util = np.mean([v["utility"] for v in fb_results[mn].values()])
    print(f"  Done: {n_items} items, mean utility = {mean_util:+.4f}")

print(f"\nFamily B sweep complete: {len(fb_results)} models")

In [ ]:
# Cell 7 — Export Raw Results to CSV (for reproducibility)
import csv

# Calibration audit CSV
cal_rows = []
for mn, items in cal_results.items():
    for iid, v in items.items():
        gold = cal_answer_key.get(iid, {})
        cal_rows.append({
            "model_name": mn, "item_id": iid,
            "question": gold.get("question", ""),
            "gold_answer": gold.get("gold_answer", ""),
            "mechanism": v["mechanism"],
            "model_answer": v["model_answer"],
            "confidence": v["confidence"],
            "is_correct": v["is_correct"],
            "brier_score": round(v["brier_score"], 4),
        })

cal_csv_path = os.path.join(OUTPUT_DIR, "calibration_item_audit.csv")
with open(cal_csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(cal_rows[0].keys()))
    w.writeheader()
    w.writerows(cal_rows)

# Family B audit CSV
fb_rows = []
for mn, items in fb_results.items():
    for iid, v in items.items():
        gold = fb_answer_key.get(iid, {})
        fb_rows.append({
            "model_name": mn, "item_id": iid,
            "question": gold.get("question", ""),
            "gold_answer": gold.get("gold_answer", ""),
            "gold_action": v["gold_action"],
            "model_decision": v["model_decision"],
            "model_answer": v["model_answer"],
            "confidence": v["confidence"],
            "is_correct": v["is_correct"],
            "utility": round(v["utility"], 4),
        })

fb_csv_path = os.path.join(OUTPUT_DIR, "family_b_item_audit.csv")
with open(fb_csv_path, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(fb_rows[0].keys()))
    w.writeheader()
    w.writerows(fb_rows)

print(f"Exported: {cal_csv_path} ({len(cal_rows)} rows)")
print(f"Exported: {fb_csv_path} ({len(fb_rows)} rows)")

---

## Results

### Calibration Leaderboard (Family A, Clean Set)

In [ ]:
# Cell 9 — Calibration Leaderboard + Reliability Diagram
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", font_scale=1.1)

model_order = sorted(cal_results.keys(), key=short_name)
colors = sns.color_palette("tab10", n_colors=len(model_order))
model_colors = {m: colors[i] for i, m in enumerate(model_order)}

# Compute per-model metrics
cal_metrics = {}
for mn in model_order:
    items = cal_results[mn]
    confs = [v["confidence"] for v in items.values()]
    corrects = [v["is_correct"] for v in items.values()]
    briers = [v["brier_score"] for v in items.values()]
    cal_metrics[mn] = {
        "accuracy": float(np.mean(corrects)),
        "mean_1_brier": float(np.mean(briers)),
        "ece": expected_calibration_error(confs, corrects),
        "overconf_rate": overconfidence_rate(confs, corrects),
        "n": len(items),
    }

# Print leaderboard
rows = []
for mn in sorted(cal_metrics, key=lambda m: -cal_metrics[m]["mean_1_brier"]):
    m = cal_metrics[mn]
    rows.append({
        "Model": short_name(mn), "Accuracy": f"{m['accuracy']:.3f}",
        "1-Brier": f"{m['mean_1_brier']:.4f}", "ECE": f"{m['ece']:.4f}",
        "Overconf": f"{m['overconf_rate']:.0%}", "n": m["n"],
    })
print("=== Calibration Leaderboard (Clean Set) ===")
print(pd.DataFrame(rows).to_string(index=False))

# --- Figure 1: Reliability Diagram ---
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Perfect calibration")
for mn in model_order:
    confs = [v["confidence"] for v in cal_results[mn].values()]
    corrects = [v["is_correct"] for v in cal_results[mn].values()]
    bins = accuracy_by_confidence_bucket(confs, corrects)
    xs = [b[0] for b in bins if b[2] > 0]
    ys = [b[1] for b in bins if b[2] > 0]
    ax.plot(xs, ys, "o-", color=model_colors[mn],
            label=short_name(mn), markersize=6)
ax.set_xlabel("Stated Confidence")
ax.set_ylabel("Observed Accuracy")
ax.set_title("Figure 1: Calibration Reliability Diagram")
ax.legend(loc="lower right", fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
fig.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig1_reliability_diagram.png"), dpi=150, bbox_inches="tight")
plt.show()

A well-calibrated model traces the diagonal. Above = underconfidence; below = overconfidence.

---

### Family B Leaderboard (Selective Abstention, Clean Set)

In [ ]:
# Cell 11 — Family B Leaderboard + Action Distribution (Figure 2)

fb_metrics = {}
fb_model_order = []
for mn in model_order:
    items = fb_results.get(mn, {})
    if len(items) < 10:
        print(f"  {short_name(mn)}: only {len(items)} FB items -- excluded")
        continue
    fb_model_order.append(mn)
    utils = [v["utility"] for v in items.values()]
    decisions = [v["model_decision"] for v in items.values()]
    golds = [v["gold_action"] for v in items.values()]
    fb_metrics[mn] = {
        "uwaa": compute_uwaa(utils),
        "mean_utility": float(np.mean(utils)),
        "action_accuracy": float(np.mean([d == g for d, g in zip(decisions, golds)])),
        "n": len(items),
    }

# Print leaderboard
rows = []
for mn in sorted(fb_metrics, key=lambda m: -fb_metrics[m]["uwaa"]):
    m = fb_metrics[mn]
    rows.append({
        "Model": short_name(mn), "UWAA": f"{m['uwaa']:.4f}",
        "Mean Util": f"{m['mean_utility']:+.4f}",
        "Action Acc": f"{m['action_accuracy']:.0%}", "n": m["n"],
    })
print("=== Family B Leaderboard (Clean Set) ===")
print(pd.DataFrame(rows).to_string(index=False))

# --- Figure 2: Action Distribution ---
actions = ["answer", "clarify", "verify", "abstain"]
action_colors = {"answer": "#4CAF50", "clarify": "#2196F3",
                 "verify": "#FF9800", "abstain": "#F44336"}

fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(fb_model_order))
for action in actions:
    vals = []
    for mn in fb_model_order:
        decisions = [v["model_decision"] for v in fb_results[mn].values()]
        vals.append(sum(1 for d in decisions if d == action) / len(decisions))
    ax.bar([short_name(m) for m in fb_model_order], vals,
           bottom=bottom, label=action, color=action_colors[action])
    bottom += np.array(vals)
ax.set_ylabel("Proportion")
ax.set_title("Figure 2: Action Distribution by Model (Family B)")
ax.legend()
fig.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig2_action_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

---

### Partial Composite MetaScore

**MetaScore (partial) = 0.60 x Calibration + 0.40 x UWAA**

This follows the two-axis framework: monitoring 60%, control 40%. The full MetaScore will include Families C-E when instrumented.

In [ ]:
# Cell 13 — Composite MetaScore

composite_rows = []
for mn in model_order:
    cal_score = cal_metrics[mn]["mean_1_brier"]
    fb_m = fb_metrics.get(mn)
    if fb_m:
        subscores = {
            "calibration": cal_score,
            "abstention_verification": fb_m["uwaa"],
        }
        meta = compute_composite_score(subscores)
        composite_rows.append({
            "Model": short_name(mn),
            "Calibration": f"{cal_score:.4f}",
            "UWAA": f"{fb_m['uwaa']:.4f}",
            "MetaScore": f"{meta:.4f}",
        })
    else:
        composite_rows.append({
            "Model": short_name(mn),
            "Calibration": f"{cal_score:.4f}",
            "UWAA": "N/A",
            "MetaScore": "N/A",
        })

print("=== Partial Composite MetaScore ===")
print(pd.DataFrame(composite_rows).to_string(index=False))

---

## Statistical Analysis

Pairwise bootstrap CIs (10,000 resamples, seed=42) with Holm-Bonferroni correction for multiple comparisons.

In [ ]:
# Cell 15 — Pairwise Bootstrap CIs + Forest Plot (Figure 3)

models = sorted(cal_results.keys())
common_items = sorted(set.intersection(
    *(set(cal_results[m].keys()) for m in models)
))

pairwise = []
all_p = {}

for m_a, m_b in combinations(models, 2):
    pair = f"{short_name(m_a)} vs {short_name(m_b)}"
    brier_a = [cal_results[m_a][i]["brier_score"] for i in common_items]
    brier_b = [cal_results[m_b][i]["brier_score"] for i in common_items]
    boot = paired_bootstrap_ci(brier_a, brier_b)

    sig = "Yes" if boot["ci_lower"] > 0 or boot["ci_upper"] < 0 else "No"
    pairwise.append({
        "Pair": pair,
        "Delta": boot["observed_diff"],
        "CI_lo": boot["ci_lower"],
        "CI_hi": boot["ci_upper"],
        "Sig": sig,
    })
    # Use whether CI excludes 0 as proxy p-value for Holm
    all_p[pair] = 0.001 if sig == "Yes" else 0.50

corrected = holm_correction(all_p)
n_sig = sum(1 for v in corrected.values() if v["significant_0.05"])

# Print table
rows = []
for pw in pairwise:
    holm_sig = corrected.get(pw["Pair"], {}).get("significant_0.05", False)
    rows.append({
        "Pair": pw["Pair"],
        "Brier Delta": f"{pw['Delta']:+.4f}",
        "95% CI": f"[{pw['CI_lo']:.4f}, {pw['CI_hi']:.4f}]",
        "Sig (Holm)": "Yes" if holm_sig else "No",
    })
print(f"=== Pairwise Calibration Comparisons (n={len(common_items)}) ===")
print(pd.DataFrame(rows).to_string(index=False))
print(f"\nHolm-Bonferroni: {n_sig}/{len(corrected)} significant at alpha=0.05")

# --- Figure 3: Brier Forest Plot ---
fig, ax = plt.subplots(figsize=(10, 5))
pairs = [pw["Pair"] for pw in pairwise]
for i, pw in enumerate(pairwise):
    ax.errorbar(pw["Delta"], i,
                xerr=[[pw["Delta"] - pw["CI_lo"]],
                      [pw["CI_hi"] - pw["Delta"]]],
                fmt="o", color="steelblue", capsize=4, markersize=6)
ax.axvline(0, color="red", linestyle="--", alpha=0.4)
ax.set_yticks(range(len(pairs)))
ax.set_yticklabels(pairs, fontsize=9)
ax.set_xlabel("Brier Score Difference (A - B)")
ax.set_title("Figure 3: Pairwise Brier Differences with 95% Bootstrap CI")
fig.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig3_brier_forest_plot.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 16 — Per-Mechanism Heatmap (Figure 4) + Bridge Analysis

# --- Figure 4: Mechanism x Model Accuracy Heatmap ---
mechanisms = sorted(set(
    v["mechanism"] for items in cal_results.values()
    for v in items.values() if v["mechanism"]
))

mech_data = []
for mech in mechanisms:
    row = {"Mechanism": mech}
    for mn in model_order:
        items = {k: v for k, v in cal_results[mn].items()
                 if v["mechanism"] == mech}
        if items:
            row[short_name(mn)] = float(np.mean(
                [v["is_correct"] for v in items.values()]))
        else:
            row[short_name(mn)] = float("nan")
    row["n"] = sum(1 for v in cal_results[model_order[0]].values()
                   if v["mechanism"] == mech)
    mech_data.append(row)

df_mech = pd.DataFrame(mech_data).set_index("Mechanism")
n_col = df_mech.pop("n")

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(df_mech, annot=True, fmt=".2f", cmap="RdYlGn",
            vmin=0, vmax=1, linewidths=0.5, ax=ax)
ax.set_title("Figure 4: Accuracy by Mechanism x Model")
fig.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig4_mechanism_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()
print("\nItems per mechanism:", dict(zip(mechanisms, n_col.values)))

# --- Bridge: Spearman confidence-accuracy correlation ---
print("\n=== Bridge: Confidence-Accuracy Correlation ===")
bridge_rows = []
for mn in model_order:
    confs = [v["confidence"] for v in cal_results[mn].values()]
    corrects = [float(v["is_correct"]) for v in cal_results[mn].values()]
    sp = spearman_with_ci(confs, corrects)
    bridge_rows.append({
        "Model": short_name(mn),
        "Spearman rho": f"{sp['rho']:.3f}",
        "p-value": f"{sp['p_value']:.4f}",
        "95% CI": f"[{sp['ci_lower']:.3f}, {sp['ci_upper']:.3f}]",
    })
print(pd.DataFrame(bridge_rows).to_string(index=False))

---

## What This Benchmark Reveals

1. **Calibration quality differs between models** -- some are systematically overconfident on wrong answers, while others maintain realistic confidence estimates.

2. **Action selection is distinct from answer quality** -- a model that scores well on calibration does not necessarily choose the right metacognitive action (answer vs. abstain vs. verify vs. clarify).

3. **Mechanism-level analysis reveals systematic failure patterns** -- models have characteristic weaknesses (anchoring traps, monitoring traps) that a global score would obscure.

4. **Monitoring and control are partially dissociable** -- the bridge analysis (Spearman rho) measures whether calibration quality predicts action-selection quality. A weak correlation means these are worth measuring separately.

---

## Limitations

- **Sample sizes**: 105 calibration items and 72 Family B items provide adequate power for moderate effects but limit per-mechanism inference.
- **Single run**: Results reflect one pass per model at temperature=0.0. Multi-run variance analysis is planned.
- **RLHF mechanism**: Only 2 clean items remain; results are exploratory.
- **Temporal sensitivity**: Some items reference time-dependent facts, flagged in the audit.
- **Models evaluated in default configuration**: Tool use is unavailable (no tool definitions provided). Chain-of-thought/reasoning varies by model and is not toggleable for all (e.g. DeepSeek V3.1). This is by design -- we measure metacognition as deployed.

## Next Steps

- Families C-E (self-correction, grounding sensitivity, control policy adaptation) are designed but not yet instrumented.
- Multi-run replication to estimate test-retest reliability.
- Full composite MetaScore when all families are scored.